# SiPM module — 60-second dose measurement

Connects to a local SiPM power supply board, prints a full
device status snapshot, then polls the output voltage for one minute and displays
voltage as a matplotlib figure.

**Prerequisites**: `pip install -e ".[dev]"` from the repo root.

In [1]:
import logging
import time

import matplotlib.pyplot as plt
import numpy as np

from nlab_modbus.core.enums import DeviceType
from nlab_modbus.maps.sipm_map import SIPM_REGISTER_MAP
from nlab_modbus.services.manager import DeviceManager

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    force=True,
)
logging.getLogger("pymodbus").setLevel(logging.CRITICAL)

## Configuration

Edit the variables below to match your setup.

In [2]:
PORT          = 'COM4'              # COM port 
DEVICE_ID     = 1                 # Modbus device address (1-16)
BAUDRATE      = 9600
DURATION_S    = 60                # measurement window in seconds
POLL_INTERVAL = 0.5               # seconds between reads (≥ 0.2 recommended)

## Connect

In [3]:
mgr = DeviceManager()
device = mgr.connect_local(PORT, DEVICE_ID, DeviceType.SIPM)

print(f"Connected: {device.connection_info()}  |  type: {device.device_type.name}")

Connected: serial://COM4:1  |  type: SIPM


## Device status snapshot

Reads all holding and input registers once and prints them with engineering units.

In [4]:
def _unit(name: str) -> str:
    spec = SIPM_REGISTER_MAP.get(name)
    return spec.unit if (spec and spec.unit) else ""


def print_status(dev) -> None:
    holding = dev.get_all_holding_registers()
    inputs  = dev.get_all_input_registers()

    col_w = max(len(k) for k in {**holding, **inputs})

    print(f"{'─' * 55}")
    print(f"  {dev.device_type.name} — {dev.connection_info()}")
    print(f"{'─' * 55}")

    print("\n  [Holding registers — read/write]")
    for name, value in holding.items():
        unit = _unit(name)
        print(f"    {name:<{col_w}}  {value:>10.4g}  {unit}")

    print("\n  [Input registers — read-only]")
    for name, value in inputs.items():
        unit = _unit(name)
        print(f"    {name:<{col_w}}  {value:>10.4g}  {unit}")

    print(f"{'─' * 55}\n")


print_status(device)

───────────────────────────────────────────────────────
  SIPM — serial://COM4:1
───────────────────────────────────────────────────────

  [Holding registers — read/write]
    rs485_mb_addr                     1  
    rs485_baud                     9600  
    vout_pwr_en                       0  
    vout_set                         40  mV
    pid_p_vout                      100  
    pid_i_vout                      100  
    sipm_comp_en                      0  
    sipm_comp_tref                   20  *C
    sipm_comp_ct                      0  mv/*C
    vout_corr                         0  
    led_drv_enable                    0  
    led_drv_period_us                 1  us
    led_drv_duration_ns              10  ns
    led_drv_imod_reg                255  
    led_drv_ibias_reg                 0  
    pass_static                   -4345  

  [Input registers — read-only]
    hardware_version                257  
    firmware_version                  2  
    vout_overload        

In [5]:
mgr.close_all()
print("Connection closed.")

Connection closed.
